In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import mysql.connector
import time
def create_connection():
    connection = mysql.connector.connect(
        host='localhost',
        database='imdb_movies',
        user='root',
        password='Bca1993@hr'
    )
    print("Connected to MySQL database")
    return connection
def insert_movie_data(connection, Ranking, Name, Release_Date, Run_Time, IMDB_Rating):
    cursor = connection.cursor( )
    
    query = """
    INSERT INTO movies (Ranking, Name, Release_Date, Run_Time, IMDB_Rating)
    VALUES (%s, %s, %s, %s, %s)
    """
    values = (Ranking, Name, Release_Date, Run_Time, IMDB_Rating)
    
    cursor.execute(query, values)
    connection.commit()

    print(f"Inserted: {Name}")
    
def scrape_and_store():

    driver = webdriver.Chrome()

    url = "https://www.imdb.com/chart/top/"
    driver.get(url)
    
    time.sleep(10)

    connection = create_connection()

    titles = driver.find_elements(By.CLASS_NAME, "ipc-title__text")

    metadata = driver.find_elements(By.CLASS_NAME, "ipc-inline-list__item")

    ratings = driver.find_elements(By.CLASS_NAME, "ipc-rating-star--rating")

    rank = 1
    meta_index = 0

    for i in range(1, min(251, len(titles))):

        try:
            name = titles[i].text.strip()
            
            if meta_index >= len(metadata):
                print("Metadata finished.")
                break
            item1 = metadata[meta_index].text.strip()
            
            if item1.isdigit() and len(item1) == 4:
                year = item1
            else:
                print("Skipping invalid year:", item1)
                meta_index += 1
                continue
            runtime = None
            if meta_index + 1 < len(metadata):
                item2 = metadata[meta_index + 1].text.strip()

                if "h" in item2 or "m" in item2:
                    runtime = item2
            if meta_index + 2 < len(metadata):
                item3 = metadata[meta_index + 2].text.strip()
                
                certificates = ["U", "A", "UA", "R", "PG", "PG-13", "12A", "15", "18", "X"]

                if item3 in certificates or item3.isalpha() or "-" in item3:
                    meta_index += 3
                else:
                    meta_index += 2
            else:
                meta_index += 2

            if i - 1 < len(ratings):
                rating = ratings[i - 1].text.strip()
            else:
                rating = None

            print(rank, name, year, runtime, rating)

            insert_movie_data(
                connection,
                rank,
                name,
                int(year),
                runtime,
                float(rating) if rating else None
            )

            rank += 1

        except Exception as e:
            print("Error:", e)
            continue

    connection.close()
    driver.quit()

    print("Finished scraping and storing data.")
    
scrape_and_store()

Connected to MySQL database
1 The Shawshank Redemption 1994 2h 22m 9.3
Inserted: The Shawshank Redemption
2 The Godfather 1972 2h 55m 9.2
Inserted: The Godfather
3 The Dark Knight 2008 2h 32m 9.1
Inserted: The Dark Knight
4 The Godfather Part II 1974 3h 22m 9.0
Inserted: The Godfather Part II
5 12 Angry Men 1957 1h 36m 9.0
Inserted: 12 Angry Men
6 The Lord of the Rings: The Return of the King 2003 3h 21m 9.0
Inserted: The Lord of the Rings: The Return of the King
7 Schindler's List 1993 3h 15m 9.0
Inserted: Schindler's List
8 The Lord of the Rings: The Fellowship of the Ring 2001 2h 58m 8.9
Inserted: The Lord of the Rings: The Fellowship of the Ring
9 Pulp Fiction 1994 2h 34m 8.8
Inserted: Pulp Fiction
10 The Good, the Bad and the Ugly 1966 2h 28m 8.8
Inserted: The Good, the Bad and the Ugly
11 The Lord of the Rings: The Two Towers 2002 2h 59m 8.8
Inserted: The Lord of the Rings: The Two Towers
12 Forrest Gump 1994 2h 22m 8.8
Inserted: Forrest Gump
Skipping invalid year: 12
13 Inceptio